# AutoMorphalyzer on three shapes, biomarker by biomarker

A close reading of one implementation. The
[main notebook](../biomarker-synthetic.ipynb) compares six implementations across thirteen shapes and answers
*which of these agree*; this one asks a narrower question about
[AutoMorphalyzer](../../../docs/projects/automorphalyzer.md) alone — **what does it actually return, on the three
simplest shapes, at each angle, beside what the geometry requires?**

Everything is under **AutoMorphalyzer's own column names**, not the catalogued ones. That is the point:
the catalogued names exist to make two programs comparable, and there is only one program here, so
the names its authors chose are the ones a reader checking against their documentation will look
for. Where a column maps onto a catalogued biomarker the mapping is shown beside it, and where the
shape settles a theoretical value that is shown too.

**The three shapes, in order of how much they ask for:**

| Shape | What it is | Why it comes first |
| --- | --- | --- |
| `straight` | one straight vessel per class, leaving the optic disc | every curvature measure must read zero and every tortuosity exactly 1 |
| `arc` | a circular arc per class | curvature is exactly 1/r, and the arc-to-chord ratio has a closed form |
| `sinusoid` | a sine wave per class | the only one of the three whose curvature changes sign |

A blank in the theory column means **this shape settles no value for that quantity** — which is not
the same as the quantity being wrong, and not the same as AutoMorphalyzer declining to answer it.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display


def repository() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").is_dir():
            return candidate
    raise RuntimeError(f"nothing above {Path.cwd()} looks like the fundus-atlas repository")


ROOT = repository()
sys.path.insert(0, str(ROOT / "src"))

from benchmarks import biomarker_synthetic as benchmark  # noqa: E402
from benchmarks.shapes import store  # noqa: E402
from biomarkers import canonical  # noqa: E402
from biomarkers.utils import catalogue  # noqa: E402

#: The one implementation this notebook reads. Change it and everything below follows.
IMPLEMENTATION = "automorphalyzer"

#: In order of how much they ask for: a straight vessel, then a curve, then a curve that changes
#: its mind about which way it bends.
SHAPES = ("straight", "arc", "sinusoid")

EVIDENCE = ROOT / "results" / benchmark.NAME / IMPLEMENTATION
STORE = ROOT / store.STORE
pd.set_option("display.width", 200, "display.max_rows", 200)

#: AutoMorph's own column name → the catalogued biomarker it answers to, or None where the
#: catalogue has no name for it. This is the adapter's claim, not the run's.
NAMES = catalogue.load(IMPLEMENTATION).declare()["names"]

TRUTH = pd.read_csv(STORE / "ground_truth.csv").set_index("key")

#: How many microns a pixel is worth, read from the manifest rather than assumed.
UM_PER_PX = float(pd.DataFrame(store.read(STORE))["um_per_px"].iloc[0])
print(f"{IMPLEMENTATION}: {len(NAMES)} columns, "
      f"{sum(1 for v in NAMES.values() if v)} of them with a catalogued name")

automorphalyzer: 66 columns, 40 of them with a catalogued name


In [2]:
def readings(shape: str) -> pd.DataFrame:
    """What AutoMorph returned for one shape: a row per column, a column per angle.

    The theoretical value is joined on last and only where the shape settles one, so a blank there
    **Every number in a row is in the implementation's own units, which are pixels.** A catalogued
    biomarker is in microns, so the theory is converted *back* onto the pixel grid before it is put
    beside figures this program actually printed. The `unit` column says what the catalogued name
    is in.

    The theoretical value is joined on last, so a blank is a statement about the fixture rather
    than about AutoMorph.
    """
    measured = pd.read_csv(EVIDENCE / f"{shape}.csv")
    rows = {}
    for own in NAMES:
        values = {}
        for _, rendering in measured.iterrows():
            values[f"{rendering['rotation']:.0f}°"] = rendering.get(f"said_{own}")
        catalogued = NAMES[own]
        truth, unit = np.nan, ""
        if catalogued:
            unit = canonical.unit(catalogued)
            keys = [k for k in TRUTH.index if k.startswith(f"{shape}-")]
            if keys and catalogued in TRUTH.columns:
                # Back onto the pixel grid: the inverse of the one conversion this repository makes.
                truth = canonical.from_pixels(
                    catalogued, TRUTH.loc[keys[0], catalogued], 1.0 / UM_PER_PX
                )
        rows[own] = {
            **values,
            "theory, in px": truth,
            "catalogued as": catalogued or "—",
            "unit": unit or "—",
        }
    frame = pd.DataFrame(rows).T
    frame.index.name = f"{IMPLEMENTATION}'s own name"
    return frame


def show(shape: str) -> pd.DataFrame:
    frame = readings(shape)
    angles = [c for c in frame.columns if c.endswith("°")]
    # laterality is a word, not a number: it shows as a blank here rather than stopping the table.
    numeric = frame[angles + ["theory, in px"]].apply(pd.to_numeric, errors="coerce")
    spread = numeric[angles].max(axis=1) - numeric[angles].min(axis=1)
    out = numeric.round(4)
    out["moves"] = spread.round(4)
    out["catalogued as"] = frame["catalogued as"]
    out["unit"] = frame["unit"]
    return out

## 1. `straight`

A single straight vessel per class, leaving the optic disc. **Everything about it is known
exactly**: the arc-to-chord ratio is 1, every curvature measure is 0, and the width is the width it
was drawn at. Nothing here is a hard measurement, which is what makes it the right place to start —
a value that is wrong on a straight line is wrong for a reason that has nothing to do with
difficulty.

In [3]:
show("straight")

,0°,15°,30°,45°,60°,75°,90°,"theory, in px",moves,catalogued as,unit
automorphalyzer's own name,,,,,,,,,,,
fractal_dimension@whole_binary,0.9683,0.9823,1.0112,0.9758,1.0008,0.9897,0.9683,1.0000,0.0429,density/box-counting/vessels,1
vessel_density@whole_binary,0.0047,0.0048,0.0048,0.0047,0.0048,0.0048,0.0047,0.0049,0.0001,density/over-image/vessels,1
average_global_calibre@whole_binary,15.8607,16.2627,18.0996,21.6394,18.1167,16.3549,15.8734,15.8025,5.7787,calibre/width/vessels/length-weighted,µm
tortuosity_distance@whole_binary,1.0010,1.0733,1.0731,1.0000,1.0753,1.0733,1.0017,1.0000,0.0753,tortuosity/hart-tau1/vessels/mean,1
tortuosity_density@whole_binary,0.2500,0.4984,0.4989,0.0000,0.5993,0.4985,0.2500,0.0000,0.5993,tortuosity/grisan-density/vessels/mean,1/µm
average_local_calibre@whole_binary,15.5090,16.6149,16.4679,16.2502,15.6963,16.6216,15.5078,NaN,1.1137,—,—
CRAE_Knudtson@whole_binary,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,—,—
CRVE_Knudtson@whole_binary,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,—,—
CRAE_Knudtson@B_binary,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,—,—


## 2. `arc`

A circular arc per class, the two sharing a subtended angle and differing in radius. The
arc-to-chord ratio is `θ / (2 sin(θ/2))` in closed form and **independent of radius**, so both
classes must return the same tortuosity while their curvatures differ. That pairing is the test:
an implementation that returns different tortuosities for the two is measuring the radius somewhere
it should not.

In [4]:
show("arc")

,0°,15°,30°,45°,60°,75°,90°,"theory, in px",moves,catalogued as,unit
automorphalyzer's own name,,,,,,,,,,,
fractal_dimension@whole_binary,1.0582,1.0987,1.0966,1.0337,1.0408,1.0644,1.0582,1.0000,0.0650,density/box-counting/vessels,1
vessel_density@whole_binary,0.0050,0.0050,0.0050,0.0050,0.0050,0.0050,0.0050,0.0050,0.0000,density/over-image/vessels,1
average_global_calibre@whole_binary,17.5331,16.9224,16.1932,16.4149,17.1292,17.4833,17.5184,15.3510,1.3399,calibre/width/vessels/length-weighted,µm
tortuosity_distance@whole_binary,1.1090,1.0888,1.0834,1.1219,1.1138,1.1268,1.1088,1.0650,0.0434,tortuosity/hart-tau1/vessels/mean,1
tortuosity_density@whole_binary,0.6613,0.7983,0.0000,0.0000,0.9797,0.9934,0.6632,0.0000,0.9934,tortuosity/grisan-density/vessels/mean,1/µm
average_local_calibre@whole_binary,15.7640,14.4989,15.1670,16.4961,16.4312,16.4819,15.5626,NaN,1.9972,—,—
CRAE_Knudtson@whole_binary,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,—,—
CRVE_Knudtson@whole_binary,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,—,—
CRAE_Knudtson@B_binary,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,—,—


## 3. `sinusoid`

A sine wave per class, and **the only one of the three whose curvature changes sign**. Its arc
length is an elliptic integral with no elementary closed form, so the theoretical value here is the
integral rather than a formula. It is the shape that settles the inflection count and Grisan's
density, both of which are zero on everything else.

In [5]:
show("sinusoid")

,0°,15°,30°,45°,60°,75°,90°,"theory, in px",moves,catalogued as,unit
automorphalyzer's own name,,,,,,,,,,,
fractal_dimension@whole_binary,1.0217,1.0335,1.0368,1.0341,1.0425,1.0562,1.0217,1.0000,0.0345,density/box-counting/vessels,1
vessel_density@whole_binary,0.0055,0.0055,0.0055,0.0055,0.0055,0.0055,0.0055,0.0055,0.0000,density/over-image/vessels,1
average_global_calibre@whole_binary,18.0566,17.9320,17.5654,17.1305,17.1644,17.6204,18.0424,15.8025,0.9261,calibre/width/vessels/length-weighted,µm
tortuosity_distance@whole_binary,1.2120,1.2110,1.2084,1.2124,1.2102,1.2107,1.2118,1.1398,0.0040,tortuosity/hart-tau1/vessels/mean,1
tortuosity_density@whole_binary,0.8734,0.9896,0.9950,0.9934,0.9938,0.9950,0.9751,0.0006,0.1216,tortuosity/grisan-density/vessels/mean,1/µm
average_local_calibre@whole_binary,16.3934,16.3655,16.3998,16.3554,16.3697,16.3678,16.3635,NaN,0.0444,—,—
CRAE_Knudtson@whole_binary,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,—,—
CRVE_Knudtson@whole_binary,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,—,—
CRAE_Knudtson@B_binary,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,—,—


## 4. What the three tables show

*Written in September 2026 against an earlier drawing of the shapes, at four angles. The tables
above were re-executed on 2026-11-25 against the store the benchmark now uses — seven angles,
6.33 µm a pixel — and they no longer show everything this section first described. Each subsection
keeps the earlier finding and says, under **Now**, what the current tables show. This notebook is
not of record ([why](../../wip/README.md)); the reading of record is the
[AutoMorph group notebook](../biomarker-synthetic-automorph.ipynb), §3.3 and §9.*

AutoMorphalyzer returned 54 columns then, three times what its ancestor
[AutoMorph](biomarker-synthetic-automorph.ipynb) did (**now** 66, against AutoMorph's 72 since
AutoMorph's own zone stage joined the benchmark), because it reports most quantities once per
**measurement zone** — `@whole`, `@B` and `@C`. Zones B and C are the conventional annuli round the
optic disc from the ARIC literature. Having three of everything is what makes these tables long and
also what makes them informative: the same quantity measured three ways is a check nothing else
here offers.

### 4.1 Twelve columns that never answer

Every `CRAE_Knudtson` and `CRVE_Knudtson` column — twelve of them, across three zones and both
classes — is **empty on all three shapes at all four angles**. The central retinal equivalents
combine the widths of several vessels crossing a ring round the disc, and none of these three
shapes puts several vessels across that ring: `straight`, `arc` and `sinusoid` each draw one vessel
per class.

That is the fixture's doing rather than a defect, and it is worth saying because an empty column
looks the same as a broken one. The shapes that would exercise these are the two `spokes` families,
which draw twelve vessels through the ring on purpose.

**Now:** all **eighteen** `CRAE_Knudtson` and `CRVE_Knudtson` columns (three zones, three vessel
classes) are still empty on all three shapes at every angle — but the shapes now settle a value for
the zone-B and zone-C artery and vein ones, by the convention that a lone vessel is passed through
Knudtson's pairing on its own (12.64 px for the artery, 18.96 for the vein). So declining is no
longer only the fixture's doing. And `AVR_B` and `AVR_C` read **1.0000** at every angle on all
three shapes, against the 0.6667 the geometry gives: where both equivalents failed, AutoMorphalyzer
divides −1 by −1 and keeps the 1 (the group notebook's §9).

### 4.2 Two calibres that disagree with each other by nine pixels (now up to seven)

It reports both a `average_global_calibre` and an `average_local_calibre` for the same vessel:

| on `straight`, vein | 0° | 30° | 60° | 90° | drawn at |
| --- | --- | --- | --- | --- | --- |
| `average_global_calibre@whole_vein` | 24.91 | 27.13 | 27.23 | 24.91 | 24 |
| `average_local_calibre@whole_vein` | **24.02** | **18.33** | **19.02** | **24.00** | 24 |

At 0° and 90° the local calibre is almost exactly right — 24.02 and 24.00 against a drawn 24 — and
the global is 4% wide. At 30° and 60° they diverge by **more than eight pixels**, a third of the
vessel's width, and the local one is now 24% narrow while the global is 13% wide.

Two numbers for one vessel, both under the implementation's own names, disagreeing by a third of
the thing being measured. Nothing in the catalogue distinguishes them, which is why only one of the
two is mapped and the other is reported unnamed.

**Now** (pixels; the geometry requires 18.96): `average_global_calibre@whole_vein` reads 19.51 at
0°, 21.77 at 30°, **26.60 at 45°**, 21.78 at 60° and 19.54 at 90°; `average_local_calibre@whole_vein`
reads 19.02, 19.57, 19.78, **16.86** and 19.01. The local one is still nearly exact at 0° and 90°
and the global one 3% wide there. They now part most at 45°, by 6.8 px — about a third of the
vessel's width — and by 4.9 px at 60°. The global calibre is still the one mapped to
`calibre/width`.

### 4.3 Grisan density reads 0.998 on a straight vessel, and the zones disagree

`tortuosity_density` must be **exactly zero** on `straight`: it counts constant-sign bends, and a
straight line has none.

| on `straight`, artery | 0° | 30° | 60° | 90° | theory |
| --- | --- | --- | --- | --- | --- |
| `tortuosity_density@whole_artery` | 0.0000 | 0.0000 | **0.9978** | 0.5000 | 0 |
| `tortuosity_density@B_artery` | 0.0000 | 0.0000 | **0.9903** | 0.0000 | 0 |
| `tortuosity_density@C_artery` | 0.0000 | 0.0000 | **0.9968** | 0.0000 | 0 |

Three zones of the same straight vessel, and at 60° all three report close to 1 for a quantity that
must be 0. The vein does the same thing at a different angle — 0.9902 at 30° in zone B, 0.0000 at
60° — so it is not one angle being unlucky. *(Figures updated to the 2026-11-25 tables; this
finding still holds, and at 75° the artery reads close to 1 in all three zones too.)*

Reading the three zones together is what makes this legible: they agree with each other closely at
every angle, which says the zone machinery is working and the quantity being fed through it is not.

### 4.4 A straight line with a fractal dimension of 0.376

`fractal_dimension@whole_artery` on `straight` reads **0.3758, 0.8841, 0.8841, 0.3758**, at the two
axis-aligned angles and the two diagonal ones respectively. A straight line has dimension 1.

Its sibling [AutoMorphClass](biomarker-synthetic-automorphclass.ipynb) returns 0.3758 on the same
shape at the same angles, and their shared ancestor AutoMorph returns 0.927. Both rewrites
introduced it and neither inherited it, which places it in whatever the two changed in common.

**Now:** no longer seen. `fractal_dimension@whole_artery` on `straight` reads 0.82–0.88 at every
angle (0.8623 at 0° and 90°), and AutoMorphClass — which now measures on the same maps as AutoMorph —
returns AutoMorph's own values, 0.84–0.93.

### 4.5 What is not checked here

Of 54 columns then, 17 carried a catalogued name and the rest were reported unnamed — every zone
variant beyond `@whole`, both calibre measures beyond the first, and every Knudtson equivalent.
**Now** 40 of 66 are named: the zone-B and zone-C variants, and the four zone Knudtson equivalents
that match their vessel class. Still unnamed are the other fourteen Knudtson columns, the nine
local calibres, the cup centre and the laterality. Unnamed
does not mean wrong; it means the catalogue has no second implementation to compare them against,
so nothing in this notebook can check them.
